# EPS Sampling Sensitivity Analysis
Generates all figures from the Sydney and Bengaluru sensitivity CSVs.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------
# Matplotlib settings
# ------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.size": 11
})

# ------------------------------------------------------------------
# Project root (edit this if running from a notebook whose cwd
# doesn't match the project folder)
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent  # notebook runs from .../notebooks, project root is one level up

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Read CSV files
# ------------------------------------------------------------------
syd_path = TABLES_DIR / "sydney_eps_draw_sensitivity.csv"
blr_path = TABLES_DIR / "bengaluru_eps_draw_sensitivity.csv"

for p in (syd_path, blr_path):
    if not p.exists():
        raise FileNotFoundError(
            f"Expected input file not found: {p}\n"
            f"Current working directory: {Path.cwd()}\n"
            f"Set PROJECT_ROOT explicitly above if your notebook's cwd "
            f"doesn't match the project folder."
        )

syd = pd.read_csv(syd_path)
blr = pd.read_csv(blr_path)

# Guard against schema drift between the two cities' CSVs
assert list(syd.columns) == list(blr.columns), (
    "Column mismatch between Sydney and Bengaluru CSVs: "
    f"{list(syd.columns)} vs {list(blr.columns)}"
)

# ==============================================================
# Figure 1: Choice Set Size Comparison
# ==============================================================
plt.figure(figsize=(6, 4))

plt.plot(
    syd["Draws (R)"],
    syd["Sampled Choice Set Size"],
    marker='o',
    linewidth=2,
    label="Sydney"
)

plt.plot(
    blr["Draws (R)"],
    blr["Sampled Choice Set Size"],
    marker='s',
    linewidth=2,
    label="Bengaluru"
)

plt.xlabel("Number of EPS Draws (R)")
plt.ylabel("Average Sampled Choice Set Size")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "choice_set_size_comparison.png",
    bbox_inches="tight"
)

plt.close()

# ==============================================================
# Figure 2: Runtime Comparison
# ==============================================================
plt.figure(figsize=(6, 4))

plt.plot(
    syd["Draws (R)"],
    syd["Run-time (s)"],
    marker='o',
    linewidth=2,
    label="Sydney"
)

plt.plot(
    blr["Draws (R)"],
    blr["Run-time (s)"],
    marker='s',
    linewidth=2,
    label="Bengaluru"
)

plt.xlabel("Number of EPS Draws (R)")
plt.ylabel("Runtime (s)")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "runtime_comparison.png",
    bbox_inches="tight"
)

plt.close()

# ==============================================================
# Figure 3 & 4: Parameter Recovery
# ==============================================================

true_values = {
    "Est beta_ivt": 0.025,
    "Est beta_wait": 0.050,
    "Est beta_walk": 0.035,
    "Est beta_transfer": 0.400
}

for df, city in [(syd, "sydney"), (blr, "bengaluru")]:

    plt.figure(figsize=(7, 5))

    for i, (column, true_value) in enumerate(true_values.items()):
        color = f"C{i}"

        plt.plot(
            df["Draws (R)"],
            df[column],
            marker='o',
            linewidth=2,
            label=column.replace("Est ", ""),
            color=color
        )

        plt.axhline(
            true_value,
            linestyle="--",
            linewidth=1,
            color=color,
            label=f"{column.replace('Est ', '')} (true)"
        )

    plt.xlabel("Number of EPS Draws (R)")
    plt.ylabel("Estimated Parameter Value")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8, ncol=2)

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"parameter_recovery_{city}.png",
        bbox_inches="tight"
    )

    plt.close()

# ==============================================================
# Figure 5: Choice Set Size vs Runtime
# ==============================================================

plt.figure(figsize=(6, 4))

plt.scatter(
    syd["Run-time (s)"],
    syd["Sampled Choice Set Size"],
    s=60,
    label="Sydney"
)

plt.scatter(
    blr["Run-time (s)"],
    blr["Sampled Choice Set Size"],
    s=60,
    label="Bengaluru"
)

for _, row in syd.iterrows():
    plt.text(
        row["Run-time (s)"],
        row["Sampled Choice Set Size"],
        str(int(row["Draws (R)"])),
        fontsize=8
    )

for _, row in blr.iterrows():
    plt.text(
        row["Run-time (s)"],
        row["Sampled Choice Set Size"],
        str(int(row["Draws (R)"])),
        fontsize=8
    )

plt.xlabel("Runtime (s)")
plt.ylabel("Average Sampled Choice Set Size")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "choice_set_vs_runtime.png",
    bbox_inches="tight"
)

plt.close()

print("=" * 60)
print("All figures saved successfully!")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print("=" * 60)

All figures saved successfully!
Output folder: D:\transit_choice_project\transit_choice_project\outputs\figures
